# deeply format the files here

We will need to remove the following columns (index included):

    (0, 'Row')
    (1, 'Column')
    (3, 'Field')
    (4, 'Object No')
    (5, 'X')
    (6, 'Y')
    (7, 'Bounding Box')
    (8, 'Position X [um]')
    (9, 'Position Y [um]')
    (10, 'Compound')
    (11, 'Concentration')
    (12, 'Cell Type')

In [16]:
import polars as pl
import os
from pathlib import Path
import duckdb as db
import shutil
from time import sleep

out_dir = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/analysis_ready_data/")
raw_format_data = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/formatted_raw_datasets")

def clean_folder(path: str) -> None:
    if os.path.exists(path):
        shutil.rmtree(path)
        sleep(0.5)
    os.makedirs(path, exist_ok=True)

In [19]:
drop_cols = [
    "Row",
    "Column",
    "Field",
    "Object No",
    "X",
    "Y",
    "Bounding Box",
    "Position X [um]",
    "Position Y [um]",
    "Compound",
    "Concentration",
    "Cell Type",
    "Cell Count"
]

clean_folder(out_dir)

for file in raw_format_data.iterdir():
    print(file.name)
    
    lf = pl.scan_csv(file, separator="\t", null_values=["", "NA", "NaN", "nan", "NULL"])
    
    lf = (
        lf
        .with_columns( # adding Well Names
            (
                pl.col("Row").map_elements(lambda x: chr(64 + x), return_dtype=pl.String) +
                pl.col("Column").cast(pl.String)
            ).alias("WellName")
        )
        .select(
            ["WellName",
            pl.all().exclude("WellName")
            ]
        )
        .drop(drop_cols)
    )
    
    out_file = out_dir / file.name
    lf.sink_csv(out_file, separator="\t")
    

2026-03-10CelPaint_ChromaNoTreatDex.tsv
2026-03-11CelPaint_PhenovNoTreatDex_HARMONY_FIXED.tsv
